# Local affine DVS motion compensation on DSEC

This notebook reproduces BA, the current default: block-wise fallback plus local 6-DoF affine event motion. Final executed CPU run: https://colab.research.google.com/drive/14mlnk92BMVWoRsIjAudiT45A4MhIl1w8

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/itsvvy/dsec-event-motion-compensation.git"
BRANCH = "main"
REPO_DIR = Path("/content/dsec-event-motion-compensation")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

%cd /content/dsec-event-motion-compensation
sys.path.insert(0, str(REPO_DIR))
%pip install -q -r requirements.txt

In [ ]:
from tests.test_synthetic import test_recovers_known_constant_translation
from tests.test_blockwise import (
    test_phase_correlation_recovers_content_motion,
    test_blockwise_estimation_recovers_two_motion_regions,
    test_short_baseline_resolves_spurious_zero_long_peak,
)
from tests.test_affine import (
    test_event_image_ecc_recovers_affine,
    test_local_affine_recovers_linear_motion_field,
)

test_recovers_known_constant_translation()
test_phase_correlation_recovers_content_motion()
test_blockwise_estimation_recovers_two_motion_regions()
test_short_baseline_resolves_spurious_zero_long_peak()
test_local_affine_recovers_linear_motion_field()
test_event_image_ecc_recovers_affine()
print("All synthetic motion-compensation tests passed.")

## Run DSEC comparison

The block-wise translation field is retained as the fallback. ECC affine regions are blended only where they satisfy correlation, displacement-bound, local-variation, and event-contrast gates.

In [ ]:
%env MPLCONFIGDIR=/content/.matplotlib
%time !python scripts/run_local_affine_demo.py --data-dir /content/dsec_data --output-dir /content/dsec_local_affine --flow-indices 2 8 12 --tile-size 64 --stride 32

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

results_dir = Path("/content/dsec_local_affine")
summary = json.loads((results_dir / "summary.json").read_text())
table = pd.DataFrame(summary["results"])
columns = [
    "flow_index", "window_ms", "event_count",
    "blockwise_median_endpoint_error_px", "affine_median_endpoint_error_px",
    "blockwise_within_2px", "affine_within_2px",
    "affine_blocks", "fallback_blocks", "runtime_s",
]
display(table[columns].round(3))
display(Image(filename=str(results_dir / "benchmark_summary.png"), width=1200))
for figure in sorted(results_dir.glob("affine_f*.png")):
    display(Image(filename=str(figure), width=1200))

## Interpretation

Local affine reduces flow error from 1.760 to 1.519 px and increases sub-2-pixel coverage from 55.6% to 63.7% while retaining block-wise translation as fallback. Remaining errors come primarily from non-linear depth/parallax and multiple motion layers. BA is the frozen default model.